# 15 — Index séparé et benchmark Facebook

Ce notebook encode les 250 chunks Facebook dans un LanceDB et un BM25 **entièrement séparés**. Il vérifie ensuite que le routeur conserve les 100 questions institutionnelles sur l’index principal, puis mesure le retrieval Facebook avec 16 questions manuelles couvrant les huit pages.

La génération Ollama n’est pas utilisée ici.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import time
import unicodedata
import re

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

def fingerprint_path(path):
    path = Path(path)
    digest = hashlib.sha256()
    if path.is_file():
        digest.update(path.read_bytes())
    elif path.is_dir():
        for child in sorted(p for p in path.rglob('*') if p.is_file()):
            digest.update(str(child.relative_to(path)).encode('utf-8'))
            digest.update(child.read_bytes())
    return digest.hexdigest()

print('Projet :', PROJECT_ROOT)

Projet : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA


In [2]:
import importlib
from chunker import Chunk
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
settings = config.facebook_scraping
facebook_chunks = [
    Chunk(**json.loads(line))
    for line in settings.chunks_candidate_full_path.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
assert facebook_chunks and all(chunk.doc_type == 'facebook_post' for chunk in facebook_chunks)

production_lancedb = config.vectorstore.uri_path
production_bm25 = config.vectorstore.bm25_index_full_path
before_lancedb = fingerprint_path(production_lancedb)
before_bm25 = fingerprint_path(production_bm25)
print('Chunks Facebook :', len(facebook_chunks))
print('Index candidat :', settings.vectorstore_candidate_full_path)

Chunks Facebook : 250
Index candidat : <PROJECT_ROOT>\data\experiments\facebook_lancedb_candidate


## Construction de l’index candidat

BGE-M3 est chargé sur le GPU et encode seulement 250 passages.

In [3]:
import numpy as np
import pandas as pd
from embeddings import EmbeddingModel
from vectorstore import VectorStoreManager

facebook_config = config.model_copy(deep=True)
facebook_config.vectorstore.uri = settings.vectorstore_candidate_uri
facebook_config.vectorstore.table_name = settings.vectorstore_candidate_table
facebook_config.vectorstore.bm25_index_path = settings.bm25_candidate_path

embedding_model = EmbeddingModel(facebook_config.embeddings)
started = time.perf_counter()
vector_batches = []
batch_size = facebook_config.embeddings.batch_size
for start in range(0, len(facebook_chunks), batch_size):
    batch = facebook_chunks[start:start + batch_size]
    vector_batches.append(embedding_model.encode([chunk.text for chunk in batch]))
facebook_vectors = np.concatenate(vector_batches, axis=0)
embedding_seconds = time.perf_counter() - started

facebook_store = VectorStoreManager(facebook_config)
facebook_store.index_all(facebook_chunks, facebook_vectors)
print('Matrice :', facebook_vectors.shape)
print('Encodage :', round(embedding_seconds, 3), 's')
print('LanceDB candidat :', facebook_store.dense.count(), 'vecteurs')

Matrice : (250, 1024)
Encodage : 4.698 s
LanceDB candidat : 250 vecteurs


In [4]:
from retriever import HybridRetriever, Reranker

facebook_reranker = Reranker(
    facebook_config.reranker.model_name,
    facebook_config.reranker.device,
)
facebook_retriever = HybridRetriever(
    facebook_config, embedding_model, facebook_store, facebook_reranker,
)
# Chauffe le cache des vecteurs et le reranker ; ce résultat n'entre pas dans les métriques.
_ = facebook_retriever.retrieve('Quelles activités récentes propose le Data Science Club ?')
print('Retriever Facebook prêt.')

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Retriever Facebook prêt.


## Validation du routeur

Le test le plus important pour ne pas dégrader le système : aucune des 100 questions institutionnelles existantes ne doit être envoyée uniquement vers Facebook.

In [5]:
from source_router import route_question

expanded_cases = [
    json.loads(line)
    for line in config.evaluation.expanded_golden_set_full_path.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
route_rows = [
    {
        'id': case.get('id'),
        'question': case['question'],
        'route': route_question(case['question']).route,
        'reason': route_question(case['question']).reason,
    }
    for case in expanded_cases
]
institutional_route_errors = [row for row in route_rows if row['route'] != 'institutional']
print('Questions institutionnelles conservées :', len(route_rows) - len(institutional_route_errors), '/', len(route_rows))
if institutional_route_errors:
    display(pd.DataFrame(institutional_route_errors))
assert not institutional_route_errors, 'Le routeur détourne une question institutionnelle.'

Questions institutionnelles conservées : 100 / 100


## Jeu Facebook manuel

Deux questions par page ciblent des informations effectivement présentes dans les snapshots. Les URLs attendues sont résolues depuis les documents candidats, puis figées dans les résultats du benchmark.

In [6]:
import frontmatter

def normalize_search(text):
    text = unicodedata.normalize('NFKD', str(text).casefold())
    text = ''.join(char for char in text if not unicodedata.combining(char))
    return re.sub(r'[^a-z0-9]+', ' ', text).strip()

candidate_docs = []
for path in settings.cleaned_candidate_full_path.glob('facebook_*.md'):
    post = frontmatter.load(path)
    candidate_docs.append({**dict(post.metadata), 'text': post.content})

def expected_url(page_key, needle=None, date_prefix=None):
    matches = [doc for doc in candidate_docs if doc['page_key'] == page_key]
    if needle:
        normalized_needle = normalize_search(needle)
        matches = [doc for doc in matches if normalized_needle in normalize_search(doc['text'])]
    if date_prefix:
        matches = [doc for doc in matches if str(doc['date']).startswith(date_prefix)]
    assert len(matches) == 1, f'Sélecteur ambigu pour {page_key}/{needle}/{date_prefix}: {len(matches)}'
    return matches[0]['url']

facebook_tests = [
    {'question': 'Où consulter le procès-verbal du jury du Master 2 en statistique agricole 2025-2026 ?', 'page': 'ENSEA_page', 'needle': 'proces-verbal du jury de recrutement', 'route': 'both'},
    {'question': 'Quel appel à communications a été publié pour CISEA 2026 ?', 'page': 'ENSEA_page', 'needle': 'appel à communications cisea 2026', 'route': 'both'},
    {'question': "Qui est la sixième et dernière finaliste du concours ENSEA d'éloquence ?", 'page': 'CDR_page', 'needle': 'sixieme et derniere finaliste', 'route': 'facebook'},
    {'question': "Que prépare le CDR pour son avenir ?", 'page': 'CDR_page', 'needle': 'avenir du cdr se dessine', 'route': 'facebook'},
    {'question': "Quelle formation de l'EJS aide à présenter et rédiger un rapport statistique ?", 'page': 'EJS_page', 'needle': 'maitriser la presentation et la redaction', 'route': 'facebook'},
    {'question': "Quel événement l'ENSEA Junior Services coorganise-t-elle en 2026 ?", 'page': 'EJS_page', 'needle': 'co-organiser une fois de plus', 'route': 'facebook'},
    {'question': 'Quel webinaire du Data Science Club porte sur Kaggle de A à Z ?', 'page': 'EDSC_page', 'needle': 'kaggle de a a z', 'route': 'facebook'},
    {'question': "Quelle conférence de l'EDSC a traité de l'IA en Afrique ?", 'page': 'EDSC_page', 'needle': 'ia en afrique opportunites', 'route': 'facebook'},
    {'question': 'Où lire la Dépêche EcoStat numéro 20 ?', 'page': 'ECOSTAT_page', 'date_prefix': '2026-06-23', 'route': 'facebook'},
    {'question': 'Où lire la Dépêche EcoStat numéro 19 ?', 'page': 'ECOSTAT_page', 'date_prefix': '2026-06-16', 'route': 'facebook'},
    {'question': "Que dit l'AES sur la Semaine culturelle 2025 ?", 'page': 'AES_page', 'needle': 'semaine culturelle 2025', 'route': 'facebook'},
    {'question': "Selon l'AES, quand ont commencé les campagnes de renouvellement des clubs et associations ?", 'page': 'AES_page', 'needle': 'depuis le 9 juin 2025', 'route': 'facebook'},
    {'question': "Quel événement de la DISE est consacré au crédit bancaire ?", 'page': 'DISE_page', 'needle': 'expertise en credit bancaire', 'route': 'facebook'},
    {'question': "Qu'est-ce que le Céleste Dîner Gala 2026 de la DISE ?", 'page': 'DISE_page', 'needle': 'une soiree d excellence d elegance et de prestige', 'route': 'facebook'},
    {'question': "Qu'est-ce que STAT-HACK 26 de la DAS ?", 'page': 'DAS_page', 'needle': 'stat-hack 26 l excellence de l analyste statisticien en action', 'route': 'facebook'},
    {'question': "Que s'est-il passé lors de la première édition de la Journée de l'Analyste ?", 'page': 'DAS_page', 'needle': 'premiere edition de la journee de l analyste a tenu ses promesses', 'route': 'facebook'},
]
for case in facebook_tests:
    case['relevant_url'] = expected_url(case['page'], case.get('needle'), case.get('date_prefix'))
    decision = route_question(case['question'])
    assert decision.route == case['route'], (case['question'], decision)
print('Cas Facebook et routage validés :', len(facebook_tests))

Cas Facebook et routage validés : 16


In [7]:
benchmark_rows = []
for case in facebook_tests:
    started = time.perf_counter()
    results = facebook_retriever.retrieve(case['question'])
    elapsed = time.perf_counter() - started
    urls = [result.url for result in results]
    rank = urls.index(case['relevant_url']) + 1 if case['relevant_url'] in urls else None
    benchmark_rows.append({
        **case,
        'rank': rank,
        'reciprocal_rank': 1 / rank if rank else 0.0,
        'hit_at_3': bool(rank and rank <= 3),
        'hit_at_5': bool(rank and rank <= 5),
        'latency_seconds': elapsed,
        'top_title': results[0].title if results else None,
        'top_url': urls[0] if urls else None,
        'retrieved_urls': urls,
    })

benchmark_df = pd.DataFrame(benchmark_rows)
facebook_metrics = pd.Series({
    'questions': len(benchmark_df),
    'mrr': benchmark_df['reciprocal_rank'].mean(),
    'hit_rate@3': benchmark_df['hit_at_3'].mean(),
    'hit_rate@5': benchmark_df['hit_at_5'].mean(),
    'mean_latency_seconds': benchmark_df['latency_seconds'].mean(),
    'p95_latency_seconds': benchmark_df['latency_seconds'].quantile(0.95),
    'route_accuracy_facebook': 1.0,
    'institutional_route_preservation': 1.0 - len(institutional_route_errors) / len(route_rows),
})
display(facebook_metrics.to_frame('valeur'))
display(benchmark_df[['question', 'page', 'rank', 'top_title', 'top_url']])

,valeur
questions,16.000000
mrr,0.937500
hit_rate@3,1.000000
hit_rate@5,1.000000
mean_latency_seconds,0.371499
p95_latency_seconds,0.419399
route_accuracy_facebook,1.000000
institutional_route_preservation,1.000000


,question,page,rank,top_title,top_url
0,Où consulter le procès-verbal du jury du Maste...,ENSEA_page,1,ensea.ci — PROCÈS-VERBAL DU JURY DE RECRUTEMEN...,https://www.facebook.com/ensea.ci/posts/pfbid0...
1,Quel appel à communications a été publié pour ...,ENSEA_page,1,ensea.ci — APPEL À COMMUNICATIONS – CISEA 2026,https://www.facebook.com/ensea.ci/posts/pfbid0...
2,Qui est la sixième et dernière finaliste du co...,CDR_page,1,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...
3,Que prépare le CDR pour son avenir ?,CDR_page,1,cdr.ensea.1 — 🎤✨ L’avenir du CDR se dessine !,https://www.facebook.com/cdr.ensea.1/posts/pfb...
4,Quelle formation de l'EJS aide à présenter et ...,EJS_page,1,people — Vous souhaitez maîtriser la présentat...,https://www.facebook.com/permalink.php?story_f...
5,Quel événement l'ENSEA Junior Services coorgan...,EJS_page,1,people — L'ENSEA Junior Services est fière de ...,https://www.facebook.com/permalink.php?story_f...
6,Quel webinaire du Data Science Club porte sur ...,EDSC_page,1,edsc.ensea — Kaggle de A à Z : Dominez les com...,https://www.facebook.com/edsc.ensea/posts/pfbi...
7,Quelle conférence de l'EDSC a traité de l'IA e...,EDSC_page,1,"edsc.ensea — Le samedi 7 mars dernier, l'ENSEA...",https://www.facebook.com/edsc.ensea/posts/pfbi...
8,Où lire la Dépêche EcoStat numéro 20 ?,ECOSTAT_page,1,DepecheEcostat — Dépêche N.20,https://www.facebook.com/DepecheEcostat/posts/...
9,Où lire la Dépêche EcoStat numéro 19 ?,ECOSTAT_page,1,DepecheEcostat — Dépêche N.19,https://www.facebook.com/DepecheEcostat/posts/...


In [8]:
failures = benchmark_df[~benchmark_df['hit_at_5']]
print('Échecs stricts à 5 :', len(failures), '/', len(benchmark_df))
if not failures.empty:
    display(failures[['question', 'page', 'relevant_url', 'retrieved_urls']])

assert facebook_store.dense.count() == len(facebook_chunks)
assert fingerprint_path(production_lancedb) == before_lancedb, 'LanceDB principal a changé.'
assert fingerprint_path(production_bm25) == before_bm25, 'BM25 principal a changé.'
print('Index Facebook séparé validé ; index institutionnel intact.')

result_path = PROJECT_ROOT / 'data' / 'experiments' / 'facebook_retrieval_benchmark.json'
result_path.write_text(json.dumps({
    'metrics': facebook_metrics.to_dict(),
    'rows': benchmark_rows,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Résultats :', result_path)

Échecs stricts à 5 : 0 / 16
Index Facebook séparé validé ; index institutionnel intact.
Résultats : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA\data\experiments\facebook_retrieval_benchmark.json


## Arrêt de validation

Sauvegardez le notebook et transmettez les métriques, le tableau des 16 questions et les éventuels échecs. Le routeur n’est pas encore branché au pipeline final.